# 216 — Validate Graph

Runs a series of checks to confirm the Layer 2 regulatory graph was ingested correctly
and is properly connected to the Layer 1 entity graph.

| Check | What it tests |
|---|---|
| 1 | Node and relationship inventory |
| 2 | Bridge test: LoanApplication → Jurisdiction → Regulation |
| 3 | Cross-reference edges |
| 4 | Sections without chunks |
| 5 | Vector index status |
| 6 | Sample vector search |
| 7 | LVR risk weight structured traversal |
| 8 | Final pass / fail summary |

In [ ]:
import sys, os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(project_root / '.env')

from src.graph.connection import Neo4jConnection

client_oai = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

conn = Neo4jConnection()
conn.connect()
print('Connected to Neo4j')

passed = []
failed = []

## Check 1 — Node and relationship inventory

In [ ]:
layer2_labels = ['Regulation', 'Section', 'Requirement', 'Threshold', 'Chunk']
layer1_labels = ['LoanApplication', 'Borrower', 'Collateral', 'Jurisdiction']

print('Node counts:')
for label in layer1_labels + layer2_labels:
    n = conn.run_query(f'MATCH (n:{label}) RETURN count(n) AS cnt')[0]['cnt']
    layer = 'L1' if label in layer1_labels else 'L2'
    print(f'  [{layer}] {label:<20} : {n}')

layer2_rels = [
    'CONTAINS_SECTION', 'CONTAINS_REQUIREMENT', 'DEFINES_THRESHOLD',
    'HAS_CHUNK', 'NEXT_CHUNK', 'CROSS_REFERENCES', 'APPLIES_TO_JURISDICTION',
    'SIMILAR_TO',
]
print('\nRelationship counts (Layer 2):')
for rel in layer2_rels:
    n = conn.run_query(f'MATCH ()-[r:{rel}]->() RETURN count(r) AS cnt')[0]['cnt']
    print(f'  {rel:<28} : {n}')

total_regs = conn.run_query('MATCH (n:Regulation) RETURN count(n) AS n')[0]['n']
if total_regs > 0:
    passed.append('Check 1: Layer 2 nodes present')
else:
    failed.append('Check 1: No Regulation nodes found')

## Check 2 — Bridge test: LoanApplication → Jurisdiction → Regulation

In [ ]:
# Confirm that Regulations are linked to the Federal jurisdiction node
result = conn.run_query(
    'MATCH (j:Jurisdiction {jurisdiction_id: $jid})<-[:APPLIES_TO_JURISDICTION]-(r:Regulation) '
    'RETURN j.name AS jurisdiction, collect(r.regulation_id) AS regulations',
    {'jid': 'JUR-AU-FED'},
)
if result:
    jname = result[0]['jurisdiction']
    regs  = result[0]['regulations']
    print(f'Jurisdiction: {jname}')
    print(f'Regulations : {regs}')
    passed.append('Check 2: Regulations linked to JUR-AU-FED')
else:
    print('\u2717 No Regulations linked to JUR-AU-FED')
    failed.append('Check 2: APPLIES_TO_JURISDICTION missing')

# Confirm LoanApplications can reach Regulations via the graph
path_result = conn.run_query(
    'MATCH (la:LoanApplication) '
    'MATCH (j:Jurisdiction {jurisdiction_id: $jid})<-[:APPLIES_TO_JURISDICTION]-(r:Regulation) '
    'RETURN count(la) AS loans, count(r) AS regulations',
    {'jid': 'JUR-AU-FED'},
)
if path_result:
    loans = path_result[0]['loans']
    regs  = path_result[0]['regulations']
    print(f'\nLoanApplications in graph : {loans}')
    print(f'Regulations via bridge    : {regs}')
    if loans > 0 and regs > 0:
        passed.append('Check 2b: LoanApplications and Regulations coexist via Jurisdiction bridge')
    else:
        failed.append('Check 2b: Bridge traversal incomplete')

## Check 3 — Cross-reference edges

In [ ]:
xrefs = conn.run_query(
    'MATCH (a:Section)-[r:CROSS_REFERENCES]->(b:Section) '
    'RETURN a.regulation_id AS from_doc, b.regulation_id AS to_doc, '
    '       r.reference_type AS ref_type, count(*) AS n '
    'ORDER BY from_doc, to_doc, ref_type'
)
if xrefs:
    print('Cross-reference edges by document pair:')
    for row in xrefs:
        print(f'  {row["from_doc"]} \u2192 {row["to_doc"]}  [{row["ref_type"]}]  {row["n"]} edges')
    passed.append('Check 3: Cross-reference edges present')
else:
    print('No cross-reference edges found (may be expected if no inter-document references were extracted).')
    passed.append('Check 3: No cross-references (acceptable)')

## Check 4 — Sections without chunks

In [ ]:
no_chunks = conn.run_query(
    'MATCH (s:Section) WHERE NOT (s)<-[:HAS_CHUNK]-(:Chunk) AND NOT (s)-[:HAS_CHUNK]->(:Chunk) '
    'RETURN s.section_id AS section_id, s.regulation_id AS regulation_id '
    'ORDER BY regulation_id, section_id'
)
if no_chunks:
    print(f'\u26a0  {len(no_chunks)} section(s) without chunks:')
    for row in no_chunks[:20]:
        print(f'  {row["regulation_id"]} / {row["section_id"]}')
    if len(no_chunks) > 20:
        print(f'  ... and {len(no_chunks) - 20} more')
else:
    print('\u2713 All sections have at least one chunk.')
    passed.append('Check 4: All sections have chunks')

if len(no_chunks) < 5:
    passed.append('Check 4: Sections without chunks within acceptable range')
else:
    failed.append(f'Check 4: {len(no_chunks)} sections missing chunks')

## Check 5 — Vector index status

In [ ]:
idx_result = conn.run_query(
    'SHOW INDEXES WHERE name = $name RETURN name, state, type',
    {'name': 'chunk_embeddings'},
)
if idx_result:
    state = idx_result[0]['state']
    itype = idx_result[0]['type']
    print(f'Index chunk_embeddings: type={itype}, state={state}')
    if state == 'ONLINE':
        passed.append('Check 5: Vector index ONLINE')
    else:
        failed.append(f'Check 5: Vector index state is {state}')
else:
    print('\u2717 Vector index chunk_embeddings not found')
    failed.append('Check 5: Vector index missing')

embedded = conn.run_query('MATCH (c:Chunk) WHERE c.embedding IS NOT NULL RETURN count(c) AS n')[0]['n']
total_chunks = conn.run_query('MATCH (c:Chunk) RETURN count(c) AS n')[0]['n']
print(f'Chunks with embeddings: {embedded} / {total_chunks}')

## Check 6 — Sample vector search

In [ ]:
TEST_QUERY = 'What is the risk weight for a standard residential mortgage with LVR between 80 and 90 percent?'

try:
    embed_response = client_oai.embeddings.create(input=[TEST_QUERY], model='text-embedding-3-small')
    query_vector   = embed_response.data[0].embedding

    results = conn.run_query(
        'CALL db.index.vector.queryNodes($index, $k, $embedding) '
        'YIELD node, score '
        'RETURN node.chunk_id AS chunk_id, node.source_document AS source, '
        '       score, left(node.text, 200) AS excerpt',
        {'index': 'chunk_embeddings', 'k': 5, 'embedding': query_vector},
    )
    print(f'Vector search results for: "{TEST_QUERY}"\n')
    for row in results:
        cid    = row['chunk_id']
        src    = row['source']
        score  = round(row['score'], 4)
        excerpt = row['excerpt']
        print(f'  [{score}] {cid} ({src})')
        print(f'  {excerpt}...')
        print()
    if results:
        passed.append('Check 6: Vector search returns results')
    else:
        failed.append('Check 6: Vector search returned no results')
except Exception as e:
    print(f'\u2717 Vector search failed: {e}')
    failed.append(f'Check 6: Vector search error: {e}')

## Check 7 — LVR risk weight structured traversal

In [ ]:
lvr_result = conn.run_query(
    'MATCH (r:Regulation {regulation_id: $rid})-[:CONTAINS_SECTION]->(s:Section) '
    '      -[:CONTAINS_REQUIREMENT]->(q:Requirement)-[:DEFINES_THRESHOLD]->(t:Threshold) '
    'WHERE t.metric = $metric '
    'RETURN t.threshold_id AS threshold_id, t.metric AS metric, '
    '       t.value AS value, t.unit AS unit, '
    '       t.condition_context AS condition_context '
    'ORDER BY t.threshold_id '
    'LIMIT 10',
    {'rid': 'APS-112', 'metric': 'risk_weight'},
)
if lvr_result:
    print(f'Risk weight thresholds from APS-112 (first {len(lvr_result)}):')
    for row in lvr_result:
        tid   = row['threshold_id']
        val   = row['value']
        unit  = row['unit']
        ctx   = row['condition_context']
        print(f'  {tid}: {val} {unit}  | context: {ctx}')
    passed.append('Check 7: APS-112 risk weight thresholds found')
else:
    print('\u26a0  No risk_weight thresholds found for APS-112')
    print('     (Run 211 first, then re-run 214 if APS-112 extraction is incomplete)')
    failed.append('Check 7: No risk_weight thresholds in APS-112')

## Check 8 — Final pass / fail summary

In [ ]:
print('=' * 60)
print('VALIDATION SUMMARY')
print('=' * 60)

for msg in passed:
    print(f'  \u2713 {msg}')
for msg in failed:
    print(f'  \u2717 {msg}')

print(f'\nPassed: {len(passed)} | Failed: {len(failed)}')

if not failed:
    print('\n\u2713 All checks passed. Layer 2 graph is ready.')
else:
    print('\n\u26a0  Some checks failed. Review the output above.')

conn.close()
print('Connection closed.')